Create Key-Pair

In [1]:
import boto3

In [ ]:
ec2=boto3.client('ec2')
ec2.describe_instances()

In [23]:
# Check current user
# !aws sts get-caller-identity
# !aws configure get region


In [ ]:
##test
ec2 = boto3.client('ec2')
response = ec2.describe_key_pairs()
for key in response['KeyPairs']:
    print(key['KeyName'])

kgptalkie


In [27]:
resp=ec2.create_key_pair(KeyName='brkeypair')

In [ ]:
resp

In [ ]:
file=open(r"path\brkeypair.pem","w")
file.write(resp['KeyMaterial'])
file.close()

Create EC2 Instance

In [ ]:
ec2.describe_instances()

response = ec2.run_instances(
    ImageId='image_id',
    MinCount=1,
    MaxCount=1,
    InstanceType='t2.micro',
    KeyName='kgptalkie',
    BlockDeviceMappings=[
        {
            "DeviceName": "/dev/xvda",
            "Ebs":{
                'DeleteOnTermination': True,
                'VolumeSize': 20
            }
        }
    ]
)

In [ ]:
response

Create Security Group

In [11]:
response = ec2.describe_security_groups()

response = ec2.create_security_group(
    GroupName="Kgptalkie",
    Description="Security group for testing"
)

In [ ]:
response

In [13]:
security_group_id = response['GroupId']
security_group_id

'sg-052eb4c5bf8f860b4'

In [15]:
# ip, port, traffic type
response = ec2.authorize_security_group_ingress(
    GroupId=security_group_id,
    IpPermissions=[
        {
            'IpProtocol': 'tcp',
            'FromPort': 22,
            'ToPort':22,
            'IpRanges':[{'CidrIp':'0.0.0.0/0'}]
        }
    ]
)   

In [ ]:
response

Attach/Detach Security Group

In [ ]:
response = ec2.describe_instances()
response

In [37]:
instance_id = response['Reservations'][0]['Instances'][0]['InstanceId']
instance_id, security_group_id

('i-0fa3674fbee90cf52', 'sg-052eb4c5bf8f860b4')

In [21]:
old_gid = response['Reservations'][0]['Instances'][0]['SecurityGroups'][0]['GroupId']

In [ ]:
ec2.modify_instance_attribute(InstanceId=instance_id, Groups=[old_gid, security_group_id])

Start, Stop and Delete EC2 instance

In [ ]:
import time

def wait_for_status(instance_id, target_status):
    while True:
        response = ec2.describe_instances(InstanceIds=instance_id)

        status = response['Reservations'][0]['Instances'][0]['State']['Name']

        if status == target_status:
            print("Instance is in {} state.".format(target_status))
            break

        time.sleep(10)

In [38]:
response = ec2.describe_instances(InstanceIds=[instance_id])
response['Reservations'][0]['Instances'][0]['State']['Name']

'terminated'

In [ ]:
def start_instance(instance_id):
    print('EC2 Instance is Starting')
    ec2.start_instances(InstanceIds=instance_id)

    wait_for_status(instance_id, 'running')

start_instance([instance_id])

In [35]:
def stop_insatances(instance_id):
    print('EC2 Instance is Stopping')
    ec2.stop_instances(InstanceIds=instance_id)

    wait_for_status(instance_id, 'stopped')

stop_insatances([instance_id])

EC2 Instance is Stopping
Instance is in stopped state.


In [36]:
def terminate_instances(instance_id):
    print('EC2 Instance is Terminating')
    ec2.terminate_instances(InstanceIds=instance_id)

    wait_for_status(instance_id, 'terminated')

terminate_instances([instance_id])

EC2 Instance is Terminating
Instance is in terminated state.
